In [ ]:
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# Loading  dataset
file_path = '/content/3) Sentiment dataset.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Dataset Shape: {df.shape}")
    print("First 5 rows:")
    print(df.head())
    print("\nColumn names:")
    print(df.columns.tolist())
except Exception as e:
    print(f"Error loading dataset: {e}")
    raise
text_column = None
label_column = None
text_column = 'Text'

Dataset Shape: (732, 15)
First 5 rows:
   Unnamed: 0.1  Unnamed: 0  \
0             0           0   
1             1           1   
2             2           2   
3             3           3   
4             4           4   

                                                Text    Sentiment  \
0   Enjoying a beautiful day at the park!        ...   Positive     
1   Traffic was terrible this morning.           ...   Negative     
2   Just finished an amazing workout! 💪          ...   Positive     
3   Excited about the upcoming weekend getaway!  ...   Positive     
4   Trying out a new recipe for dinner tonight.  ...   Neutral      

             Timestamp            User     Platform  \
0  2023-01-15 12:30:00   User123          Twitter     
1  2023-01-15 08:45:00   CommuterX        Twitter     
2  2023-01-15 15:45:00   FitnessFan      Instagram    
3  2023-01-15 18:20:00   AdventureX       Facebook    
4  2023-01-15 19:55:00   ChefCook        Instagram    

                            

In [ ]:
possible_label_cols = ['sentiment', 'label', 'category', 'class']
for col in possible_label_cols:
    if col in df.columns:
        label_column = col
        break

if label_column is None and len(df.columns) > 1:
    for col in df.columns:
        if col != text_column and df[col].nunique() < df.shape[0] / 2:
            label_column = col
            break
    if label_column is None:
        if text_column is not None and list(df.columns).index(text_column) + 1 < len(df.columns):
            label_column = df.columns[list(df.columns).index(text_column) + 1]
            print(f"Could not find common label column name. Using column '{label_column}' as label (next to text column).")
        else:
            print("Could not infer a suitable label column.")

if text_column is None or label_column is None:
    raise ValueError("Could not infer text and label columns. Please inspect your CSV and manually set 'text_column' and 'label_column'.")

print(f"\nUsing text column: '{text_column}'")
print(f"Using label column: '{label_column}'")


Using text column: 'Text'
Using label column: 'Sentiment'


In [ ]:
# Droping  missing values
df.dropna(subset=[text_column, label_column], inplace=True)
print(f"\nShape after dropping: {df.shape}")


Shape after dropping: (732, 15)


In [ ]:
# Downloading NLTK data
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Initializing lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    tokens = word_tokenize(text)
    # Removing stopwords, non-alphabetic tokens and lemmatize
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(processed_tokens)

print("\nPreprocessing text data...")
df['processed_text'] = df[text_column].apply(preprocess_text)
print("Text preprocessing complete.")
print("First 5 rows of processed text:")
print(df['processed_text'].head())


Preprocessing text data...
Text preprocessing complete.
First 5 rows of processed text:
0         enjoying beautiful day park
1            traffic terrible morning
2            finished amazing workout
3    excited upcoming weekend getaway
4    trying new recipe dinner tonight
Name: processed_text, dtype: object


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Spliting data into training and testing sets
X = df['processed_text']
y = df[label_column]

In [ ]:
# Identifying and removing classes with only one member for stratified split
class_counts = y.value_counts()
rp_classes = class_counts[class_counts < 2].index # Rare population classes

if not rp_classes.empty:
    print(f"\nRemoving {len(rp_classes)} rare classes with single members: {list(rp_classes)}")
# Filtering out rows corresponding to rare classes
    df_filtered = df[~df[label_column].isin(rp_classes)]
    X = df_filtered['processed_text']
    y = df_filtered[label_column]
    print(f"Shape after removing rare classes: {df_filtered.shape}")



Removing 145 rare classes with single members: [' Celestial Wonder ', ' Intimidation    ', ' Helplessness    ', ' Positive ', ' Anxiety         ', ' Boredom ', ' Indifference ', ' Disgust ', ' Relief ', ' Whispers of the Past ', " Ocean's Freedom ", ' Runway Creativity ', ' Anxiety   ', ' Ambivalence ', ' Helplessness ', " Nature's Beauty ", ' Creative Inspiration ', ' Curiosity   ', ' Loneliness      ', ' Elation   ', ' Compassion', ' Pride         ', ' Happiness     ', ' Amusement     ', ' JoyfulReunion ', ' Anticipation  ', ' Blessed       ', ' Appreciation  ', ' Nostalgia     ', ' Confidence    ', ' Surprise      ', ' Wonderment    ', ' Optimism      ', ' Motivation    ', ' Excitement    ', ' Satisfaction  ', ' Despair      ', ' Intimidation ', ' Awe           ', ' Nostalgia      ', ' Thrill        ', ' Calmness      ', ' Overwhelmed   ', ' Gratitude   ', ' Bittersweet ', ' Curiosity     ', ' Admiration    ', ' Overjoyed     ', ' Inspiration   ', ' Jealousy    ', ' Numbness ', ' E

In [ ]:
# Determining if stratified split is possible
n_samples = len(y)
test_size_ratio = 0.2
n_test_samples = int(n_samples * test_size_ratio) # Calculating the number of samples in the test set
n_unique_classes = y.nunique()

if n_unique_classes > 1 and n_test_samples >= n_unique_classes:
    print(f"\nSplitting data into training and testing sets with stratification ({n_unique_classes} classes, {n_test_samples} test samples).")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size_ratio, random_state=42, stratify=y)
else:
    if n_unique_classes <= 1:
        print("Warning: Only one unique class found in the label column. Stratification skipped.")
    else:
        print(f"Warning: Cannot perform stratified split. Number of test samples ({n_test_samples}) is less than the number of unique classes ({n_unique_classes}). Stratification skipped.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size_ratio, random_state=42)

print(f"\nTraining data size: {len(X_train)}")
print(f"Testing data size: {len(X_test)}")


Training data size: 469
Testing data size: 118


In [ ]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limit features to 5000 for efficiency
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nTF-IDF vectorization complete. Shape of X_train_tfidf: {X_train_tfidf.shape}")


TF-IDF vectorization complete. Shape of X_train_tfidf: (469, 1724)


In [ ]:
# Training  Logistic Regression model
print("\nTraining Logistic Regression model")
model = LogisticRegression(max_iter=1000, solver='liblinear') # Increased max_iter for convergence
model.fit(X_train_tfidf, y_train)
print("Model training complete.")


Training Logistic Regression model
Model training complete.


In [ ]:
# Evaluating the model
print("\nEvaluating model performance")
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
# Use 'weighted' average for multi-class classification or imbalanced datasets
# zero_division=0 prevents errors if a class has no predicted samples.
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"\n Model Evaluation ")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


Evaluating model performance

 Model Evaluation 
Accuracy: 0.1186
Precision: 0.0515
Recall: 0.1186
F1-Score: 0.0503


In [ ]:
# Example Prediction
print("\nExample Prediction ")
sample_text = "This is a great movie! I loved every part of it, highly recommended."
processed_sample_text = preprocess_text(sample_text)
sample_text_tfidf = tfidf_vectorizer.transform([processed_sample_text])
prediction = model.predict(sample_text_tfidf)
print(f"Sample text: '{sample_text}'")
print(f"Processed text: '{processed_sample_text}'")
print(f"Predicted category: {prediction[0]}")

sample_text_bad = "This movie was terrible, absolutely horrible. I hated it."
processed_sample_text_bad = preprocess_text(sample_text_bad)
sample_text_tfidf_bad = tfidf_vectorizer.transform([processed_sample_text_bad])
prediction_bad = model.predict(sample_text_tfidf_bad)
print(f"\nSample text: '{sample_text_bad}'")
print(f"Processed text: '{processed_sample_text_bad}'")
print(f"Predicted category: {prediction_bad[0]}")


Example Prediction 
Sample text: 'This is a great movie! I loved every part of it, highly recommended.'
Processed text: 'great movie loved every part highly recommended'
Predicted category:  Joy 

Sample text: 'This movie was terrible, absolutely horrible. I hated it.'
Processed text: 'movie terrible absolutely horrible hated'
Predicted category:  Positive  
